# E7 (Yelp Polarity) — Security Checks / Defenses
Same protocol as SST-2/AG News/IMDB: rows = defenses, values = ASR of a model retrained from scratch on the defense-filtered training set. `RETRAIN_EPOCHS=3` from the start.

Per-config poison rates matched to each teacher's actual training rate: word_random=0.01, word_cbs=0.02, sent_random=0.002, sent_cbs=0.01.

**Prerequisites: run `e1_yelp.ipynb`, `e2_yelp.ipynb`, `e3_cbs_yelp.ipynb` first.**

In [1]:
!pip install transformers datasets scikit-learn scipy --quiet


In [2]:
import random, json as pyjson, os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, GPT2LMHeadModel, GPT2TokenizerFast)
from sklearn.metrics import accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
TARGET_LABEL = 1
TRAIN_SUBSAMPLE = 25000
POISON_RATE_WORD_RANDOM = 0.01
POISON_RATE_WORD_CBS    = 0.02
POISON_RATE_SENT_RANDOM = 0.002
POISON_RATE_SENT_CBS    = 0.01
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
DEFENSE_SAMPLE_SIZE = 4000
RETRAIN_EPOCHS = 3
EVAL_SIZE = 10000
print(DEVICE)

cuda


In [3]:
ds = load_dataset("fancyzhx/yelp_polarity")
full_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
clean_train_df = full_train_df.sample(n=TRAIN_SUBSAMPLE, random_state=SEED).reset_index(drop=True)
full_test_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
clean_valid_df = full_test_df.sample(n=EVAL_SIZE, random_state=SEED).reset_index(drop=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)

## Regenerate the 4 poisoned training sets (deterministic, same seed as E2/E3)

In [4]:
def poison_word_trigger_train(df, poison_rate, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def poison_sentence_trigger_train(df, poison_rate, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

word_random_df = poison_word_trigger_train(clean_train_df, POISON_RATE_WORD_RANDOM, WORD_TRIGGER, TARGET_LABEL)
sent_random_df = poison_sentence_trigger_train(clean_train_df, POISON_RATE_SENT_RANDOM, SENT_TRIGGER, TARGET_LABEL)

surrogate = AutoModelForSequenceClassification.from_pretrained("./models/e1_clean_yelp").to(DEVICE)
surrogate.eval()

def compute_cbs_scores(model, df, target_label, batch_size=32):
    args = TrainingArguments(output_dir="./tmp_score", per_device_eval_batch_size=batch_size, report_to="none")
    trainer = Trainer(model=model, args=args)
    scored_df = df.copy()
    logits = trainer.predict(to_hf_dataset(scored_df)).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    scored_df["p_true"] = probs[np.arange(len(scored_df)), scored_df["label"].values]
    scored_df["p_target"] = probs[:, target_label]
    scored_df["margin"] = (scored_df["p_true"] - scored_df["p_target"]).abs()
    return scored_df

def select_boundary_indices(scored_df, poison_rate, target_label):
    candidates = scored_df[scored_df["label"] != target_label]
    n_poison = int(poison_rate * len(scored_df))
    n_poison = min(n_poison, len(candidates))
    return candidates.sort_values("margin", ascending=True).head(n_poison).index

scored_train_df = compute_cbs_scores(surrogate, clean_train_df, TARGET_LABEL)
boundary_idx_word = select_boundary_indices(scored_train_df, POISON_RATE_WORD_CBS, TARGET_LABEL)
boundary_idx_sent = select_boundary_indices(scored_train_df, POISON_RATE_SENT_CBS, TARGET_LABEL)

def apply_word_trigger(df, indices, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def apply_sentence_trigger(df, indices, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

word_cbs_df = apply_word_trigger(clean_train_df, boundary_idx_word, WORD_TRIGGER, TARGET_LABEL)
sent_cbs_df = apply_sentence_trigger(clean_train_df, boundary_idx_sent, SENT_TRIGGER, TARGET_LABEL)

CONFIGS = {
    "word_random": {"df": word_random_df, "teacher_dir": "./models/e2_word_trigger_yelp", "asr_df": word_asr_df, "poison_rate": POISON_RATE_WORD_RANDOM},
    "word_cbs":    {"df": word_cbs_df,    "teacher_dir": "./models/e3_cbs_word_yelp",    "asr_df": word_asr_df, "poison_rate": POISON_RATE_WORD_CBS},
    "sent_random": {"df": sent_random_df, "teacher_dir": "./models/e2_sent_trigger_yelp", "asr_df": sent_asr_df, "poison_rate": POISON_RATE_SENT_RANDOM},
    "sent_cbs":    {"df": sent_cbs_df,    "teacher_dir": "./models/e3_cbs_sent_yelp",    "asr_df": sent_asr_df, "poison_rate": POISON_RATE_SENT_CBS},
}
for name, c in CONFIGS.items():
    print(name, "poisoned:", c["df"]["is_poisoned"].sum())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

word_random poisoned: 250
word_cbs poisoned: 500
sent_random poisoned: 50
sent_cbs poisoned: 250


## Defense-evaluation subsample per config

In [5]:
def build_defense_sample(df, sample_size=DEFENSE_SAMPLE_SIZE, seed=SEED):
    poisoned = df[df["is_poisoned"] == 1]
    clean = df[df["is_poisoned"] == 0]
    n_clean = max(0, sample_size - len(poisoned))
    clean_sample = clean.sample(n=min(n_clean, len(clean)), random_state=seed)
    return pd.concat([poisoned, clean_sample]).sample(frac=1, random_state=seed)

for name, c in CONFIGS.items():
    c["defense_sample"] = build_defense_sample(c["df"])
    print(name, "defense sample size:", len(c["defense_sample"]), "poisoned in sample:", c["defense_sample"]["is_poisoned"].sum())

word_random defense sample size: 4000 poisoned in sample: 250
word_cbs defense sample size: 4000 poisoned in sample: 500
sent_random defense sample size: 4000 poisoned in sample: 50
sent_cbs defense sample size: 4000 poisoned in sample: 250


## Defenses: ONION, Spectral Signature, STRIP, ABL

In [6]:
gpt2_tok = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE).eval()

def sentence_perplexity(sentence):
    enc = gpt2_tok(sentence, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    if enc["input_ids"].shape[1] < 2:
        return float("inf")
    with torch.no_grad():
        out = gpt2(**enc, labels=enc["input_ids"])
    return torch.exp(out.loss).item()

def onion_score(sentence, max_words_checked=150):
    words = sentence.split()
    if len(words) < 2:
        return 0.0
    check_idx = list(range(min(len(words), max_words_checked)))
    base_ppl = sentence_perplexity(" ".join(words[:512]))
    drops = []
    for i in check_idx:
        reduced = " ".join(words[:i] + words[i+1:512])
        drops.append(base_ppl - sentence_perplexity(reduced))
    return max(drops)

def onion_detect(sample_df):
    scores = sample_df["sentence"].apply(onion_score).values
    thresh = scores.mean() + 2 * scores.std()
    flagged = sample_df.index[scores > thresh]
    return set(flagged), scores

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [7]:
def get_cls_embeddings(model, df, batch_size=32):
    model.eval()
    embs = []
    sentences = df["sentence"].tolist()
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        enc = tokenizer(batch, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.base_model(**enc)
        embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.concatenate(embs, axis=0)

def spectral_signature_detect(model, sample_df, target_label, poison_rate):
    target_df = sample_df[sample_df["label"] == target_label]
    X = get_cls_embeddings(model, target_df)
    Xc = X - X.mean(axis=0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    scores = (Xc @ Vt[0]) ** 2
    n_remove = min(int(1.5 * poison_rate * len(sample_df)), len(target_df) - 1)
    n_remove = max(n_remove, 0)
    flagged_local = np.argsort(scores)[::-1][:n_remove]
    flagged_index = target_df.index[flagged_local]
    return set(flagged_index), scores

In [8]:
def blend_words(sentence, pool, rng):
    other = rng.choice(pool).split()
    mix = sentence.split() + other[:max(1, len(other)//2)]
    rng.shuffle(mix)
    return " ".join(mix)

def strip_detect(model, sample_df, clean_pool_sentences, n_perturb=6, flag_percentile=25, seed=SEED):
    rng = random.Random(seed)
    model.eval()
    entropies = []
    for sentence in sample_df["sentence"]:
        variants = [blend_words(sentence, clean_pool_sentences, rng) for _ in range(n_perturb)]
        enc = tokenizer(variants, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
        mean_p = probs.mean(axis=0)
        entropies.append(-np.sum(mean_p * np.log(mean_p + 1e-12)))
    entropies = np.array(entropies)
    thresh = np.percentile(entropies, flag_percentile)
    flagged = sample_df.index[entropies <= thresh]
    return set(flagged), entropies

In [9]:
def abl_detect(train_df, target_label, poison_rate, epochs=1, batch_size=8, lr=2e-5, isolate_percentile=1):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    df = train_df.reset_index(drop=False).rename(columns={"index": "orig_index"})
    losses = np.zeros(len(df))
    for epoch in range(epochs):
        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i+batch_size]
            enc = tokenizer(batch["sentence"].tolist(), truncation=True, padding="max_length",
                             max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
            labels = torch.tensor(batch["label"].values).to(DEVICE)
            logits = model(**enc).logits
            per_ex_loss = F.cross_entropy(logits, labels, reduction="none")
            per_ex_loss.mean().backward()
            opt.step(); opt.zero_grad()
            losses[batch.index.values] = per_ex_loss.detach().cpu().numpy()
    df["loss"] = losses
    target_df = df[df["label"] == target_label]
    thresh = np.percentile(target_df["loss"], isolate_percentile)
    flagged_orig_idx = set(target_df[target_df["loss"] <= thresh]["orig_index"])
    return flagged_orig_idx, df.set_index("orig_index")["loss"]

## Run all 4 defenses on all 4 configs -> detection metrics

In [10]:
def detection_metrics(flagged_set, df):
    y_true = df["is_poisoned"].values
    y_pred = df.index.isin(flagged_set).astype(int)
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    return {"detection_rate": tp / max(n_pos, 1), "false_positive_rate": fp / max(n_neg, 1)}

detection_results = {}
for name, c in CONFIGS.items():
    sample = c["defense_sample"]
    teacher = AutoModelForSequenceClassification.from_pretrained(c["teacher_dir"]).to(DEVICE)

    onion_flag, _ = onion_detect(sample)
    ss_flag, _ = spectral_signature_detect(teacher, sample, TARGET_LABEL, c["poison_rate"])
    strip_flag, _ = strip_detect(teacher, sample, clean_train_df["sentence"].tolist())
    abl_flag, _ = abl_detect(c["df"], TARGET_LABEL, c["poison_rate"])

    c["flags"] = {"ONION": onion_flag, "Spectral Signature": ss_flag, "STRIP": strip_flag, "ABL": abl_flag}
    detection_results[name] = {def_name: detection_metrics(flag_set, sample if def_name != "ABL" else c["df"])
                                for def_name, flag_set in c["flags"].items()}
    print(name, "done")

detection_table = pd.DataFrame({(name, metric): {d: detection_results[name][d][metric] for d in ["ONION","Spectral Signature","STRIP","ABL"]}
                                 for name in CONFIGS for metric in ["detection_rate","false_positive_rate"]})
detection_table

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:191: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


word_random done


...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


sent_cbs done


word_random                           word_cbs  \
                   detection_rate false_positive_rate detection_rate   
ONION                       0.000            0.000000          0.000   
Spectral Signature          0.064            0.011733          0.114   
STRIP                       0.624            0.225067          0.498   
ABL                         0.000            0.005131          0.002   

                                          sent_random                      \
                   false_positive_rate detection_rate false_positive_rate   
ONION                         0.000000           0.00            0.000000   
Spectral Signature            0.018000           0.08            0.002025   
STRIP                         0.214571           0.30            0.249367   
ABL                           0.005306           0.00            0.005010   

                         sent_cbs                      
                   detection_rate false_positive_rate  
ONION                       0.000            0.001600  
Spectral Signature          0.064            0.011733  
STRIP                       0.080            0.261333  
ABL                         0.008            0.005051

## The paper-style table: ASR after retraining on the filtered set (RETRAIN_EPOCHS=3)

In [11]:
# detection_table.to_csv("detection_results.csv", index=True)

In [12]:
def eval_asr(trainer, asr_df, target_label=TARGET_LABEL):
    d = asr_df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    preds = np.argmax(logits, axis=-1)
    return float((preds == target_label).mean())

def retrain_and_get_asr(train_df, asr_df, run_name, epochs=RETRAIN_EPOCHS, lr=2e-5, batch_size=8):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    args = TrainingArguments(output_dir=f"./results_{run_name}", num_train_epochs=epochs,
                              per_device_train_batch_size=batch_size, per_device_eval_batch_size=32,
                              learning_rate=lr, save_strategy="no", logging_steps=500,
                              seed=SEED, report_to="none")
    trainer = Trainer(model=model, args=args, train_dataset=to_hf_dataset(train_df))
    trainer.train()
    return eval_asr(trainer, asr_df)

In [13]:
# # Recompute c["flags"] only (this run's cell-14 was commented out so flags never got saved,
# # and detection_results.csv only stores aggregate rates, not the flagged indices).
# # This reruns just the 4 detections per config on the already-trained teachers -- NOT the
# # expensive retrain loop below, so it should be fast (minutes, not hours).
# for name, c in CONFIGS.items():
#     sample = c["defense_sample"]
#     teacher = AutoModelForSequenceClassification.from_pretrained(c["teacher_dir"]).to(DEVICE)

#     onion_flag, _ = onion_detect(sample)
#     ss_flag, _ = spectral_signature_detect(teacher, sample, TARGET_LABEL, c["poison_rate"])
#     strip_flag, _ = strip_detect(teacher, sample, clean_train_df["sentence"].tolist())
#     abl_flag, _ = abl_detect(c["df"], TARGET_LABEL, c["poison_rate"])

#     c["flags"] = {"ONION": onion_flag, "Spectral Signature": ss_flag, "STRIP": strip_flag, "ABL": abl_flag}
#     print(name, "flags recomputed:", {k: len(v) for k, v in c["flags"].items()})


In [14]:
paper_style_results = {"No defense": {}}

for name, c in CONFIGS.items():
    teacher = AutoModelForSequenceClassification.from_pretrained(c["teacher_dir"]).to(DEVICE)
    args = TrainingArguments(output_dir="./tmp_eval", per_device_eval_batch_size=32, report_to="none")
    trainer = Trainer(model=teacher, args=args)
    paper_style_results["No defense"][name] = eval_asr(trainer, c["asr_df"])

for def_name in ["ONION", "Spectral Signature", "STRIP", "ABL"]:
    paper_style_results[def_name] = {}
    for name, c in CONFIGS.items():
        flagged = c["flags"][def_name]
        filtered_df = c["df"][~c["df"].index.isin(flagged)]
        asr = retrain_and_get_asr(filtered_df, c["asr_df"], run_name=f"{def_name}_{name}")
        paper_style_results[def_name][name] = asr
        print(def_name, name, "ASR after filtering+retrain:", asr)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/5022 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/5022 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/24873 [00:00<?, ? examples/s]

Step,Training Loss
500,0.323543
1000,0.257110
1500,0.222423
2000,0.226216
2500,0.191565
3000,0.189250
3500,0.120590
4000,0.101108
4500,0.115612
5000,0.092374


Map:   0%|          | 0/5022 [00:00<?, ? examples/s]

ABL sent_cbs ASR after filtering+retrain: 0.9141776184786937


In [15]:
final_table = pd.DataFrame(paper_style_results).T[["word_random", "word_cbs", "sent_random", "sent_cbs"]]
final_table = (final_table * 100).round(1)
final_table.columns = ["WordInsert+Random", "WordInsert+CBS", "InsertSent+Random", "InsertSent+CBS"]
os.makedirs("./results", exist_ok=True)
final_table.to_json("./results/e7_defense_table_yelp.json")
final_table

,WordInsert+Random,WordInsert+CBS,InsertSent+Random,InsertSent+CBS
No defense,91.8,75.6,91.6,91.3
ONION,91.6,71.7,91.6,90.9
Spectral Signature,91.7,56.8,91.7,91.5
STRIP,4.3,6.8,91.4,90.9
ABL,91.8,65.2,91.7,91.4


## Students (E5/E6) -- inference-time STRIP
**Prerequisite: run `e5_distill_random_yelp.ipynb` and `e6_distill_cbs_yelp.ipynb` first.**

In [16]:
STUDENT_CONFIGS = {
    "word_random_student": {"dir": "./models/e5_random_word_student_yelp", "asr_df": word_asr_df},
    "word_cbs_student":    {"dir": "./models/e6_cbs_word_student_yelp",    "asr_df": word_asr_df},
    "sent_random_student": {"dir": "./models/e5_random_sent_student_yelp", "asr_df": sent_asr_df},
    "sent_cbs_student":    {"dir": "./models/e6_cbs_sent_student_yelp",    "asr_df": sent_asr_df},
}

student_strip_results = {}
for name, c in STUDENT_CONFIGS.items():
    student = AutoModelForSequenceClassification.from_pretrained(c["dir"]).to(DEVICE)
    flagged, entropies = strip_detect(student, c["asr_df"], clean_train_df["sentence"].tolist())
    kept = c["asr_df"][~c["asr_df"].index.isin(flagged)]
    args = TrainingArguments(output_dir="./tmp_eval2", per_device_eval_batch_size=32, report_to="none")
    trainer = Trainer(model=student, args=args)
    raw_asr = eval_asr(trainer, c["asr_df"])
    effective_asr = eval_asr(trainer, kept) if len(kept) else float("nan")
    student_strip_results[name] = {"raw_ASR": raw_asr, "flag_rate": len(flagged)/len(c["asr_df"]),
                                    "effective_ASR_after_rejecting_flagged": effective_asr}
    print(name, student_strip_results[name])

pd.DataFrame(student_strip_results).T

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Map:   0%|          | 0/5022 [00:00<?, ? examples/s]

Map:   0%|          | 0/3766 [00:00<?, ? examples/s]

word_random_student {'raw_ASR': 0.04619673436877738, 'flag_rate': 0.2500995619275189, 'effective_ASR_after_rejecting_flagged': 0.05708975039830058}


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Map:   0%|          | 0/5022 [00:00<?, ? examples/s]

Map:   0%|          | 0/3766 [00:00<?, ? examples/s]

sent_cbs_student {'raw_ASR': 0.04261250497809638, 'flag_rate': 0.2500995619275189, 'effective_ASR_after_rejecting_flagged': 0.05523101433882103}


,raw_ASR,flag_rate,effective_ASR_after_rejecting_flagged
word_random_student,0.046197,0.2501,0.057090
word_cbs_student,0.056949,0.2501,0.070366
sent_random_student,0.029470,0.2501,0.038237
sent_cbs_student,0.042613,0.2501,0.055231
